# 🧬 AlphaHRD Structural Validation
## Boltz-2 + MutPred-PPI Pipeline

**Runtime:** Select **T4 GPU** (Runtime → Change runtime type → T4 GPU)

**Estimated time:** ~30-45 min total

This notebook:
1. Installs Boltz-2 and MutPred-PPI
2. Predicts 6 HRR protein complex structures
3. Runs MutPred-PPI to score PPI disruption for 315 variant-complex pairs
4. Exports results for manuscript figures

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1: CHECK GPU AND INSTALL DEPENDENCIES
# ═══════════════════════════════════════════════════════════
!nvidia-smi | head -5
print()

# Install Boltz-2
!pip install -q boltz

# Install MutPred-PPI dependencies
!pip install -q torch torchvision torchaudio
!pip install -q biopython gemmi

# Clone MutPred-PPI
!git clone -q https://github.com/rosstewart/MutPred-PPI.git 2>/dev/null || echo 'Already cloned'
!cd MutPred-PPI && pip install -q -r src/requirements.txt 2>/dev/null

print('\n✅ All dependencies installed')

Mon Mar 16 08:52:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 17.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2: UPLOAD THE TAR.GZ FILE
# ═══════════════════════════════════════════════════════════
# Option A: Upload from your computer
from google.colab import files
print('Upload alphahrd_structural_inputs.tar.gz:')
uploaded = files.upload()

# Extract
!tar xzf alphahrd_structural_inputs.tar.gz
!ls structures/boltz_input/
print('\n✅ Files extracted')

Upload alphahrd_structural_inputs.tar.gz:


Saving alphahrd_structural_inputs.tar.gz to alphahrd_structural_inputs.tar.gz
ATM_NBN_af3.json      BRCA1_PALB2_af3.json  MRE11_NBN_af3.json
ATM_NBN.yaml	      BRCA1_PALB2.yaml	    MRE11_NBN.yaml
BRCA1_BARD1_af3.json  BRCA2_RAD51_af3.json  PALB2_BRCA2_af3.json
BRCA1_BARD1.yaml      BRCA2_RAD51.yaml	    PALB2_BRCA2.yaml

✅ Files extracted


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3: PREDICT COMPLEX STRUCTURES WITH BOLTZ-2
# ═══════════════════════════════════════════════════════════
# This takes ~5-10 min per complex on T4
# Total: ~30-60 min for 6 complexes

import os, glob, time

yaml_files = sorted(glob.glob('structures/boltz_input/*.yaml'))
print(f'Complexes to predict: {len(yaml_files)}')
for f in yaml_files:
    print(f'  {os.path.basename(f)}')

os.makedirs('structures/boltz_output', exist_ok=True)

t0 = time.time()
for yaml_file in yaml_files:
    name = os.path.basename(yaml_file).replace('.yaml', '')
    # Skip AF3 JSON files
    if '_af3' in name:
        continue

    outdir = f'structures/boltz_output/{name}'
    if os.path.exists(outdir) and any(f.endswith('.pdb') for f in os.listdir(outdir) if os.path.isfile(os.path.join(outdir, f))):
        print(f'  {name}: already predicted, skipping')
        continue

    print(f'\n  Predicting {name}...')
    t1 = time.time()

    !boltz predict "{yaml_file}" \
        --use_msa_server \
        --out_dir "{outdir}" \
        --output_format pdb \
        --recycling_steps 3 \
        --diffusion_samples 1

    elapsed = time.time() - t1
    print(f'  {name}: done in {elapsed:.0f}s')

total = time.time() - t0
print(f'\n✅ All structures predicted in {total:.0f}s ({total/60:.1f} min)')

# List outputs
for root, dirs, fls in os.walk('structures/boltz_output'):
    for f in fls:
        if f.endswith('.pdb') or f.endswith('.cif'):
            fp = os.path.join(root, f)
            print(f'  {fp} ({os.path.getsize(fp):,} bytes)')

Complexes to predict: 0

✅ All structures predicted in 0s (0.0 min)


In [ ]:
import os, glob, shutil

# Fix: copy PDBs with correct names from Boltz output
os.makedirs('structures/pdb_final', exist_ok=True)

for pdb in glob.glob('structures/boltz_output/*/predictions/**/*.pdb', recursive=True):
    # Get complex name from path
    parts = pdb.split('/')
    for p in parts:
        if p in ['ATM_NBN','BRCA1_BARD1','BRCA1_PALB2','BRCA2_RAD51','MRE11_NBN','PALB2_BRCA2']:
            dest = f'structures/pdb_final/{p}.pdb'
            shutil.copy(pdb, dest)
            print(f'  {p}.pdb ({os.path.getsize(dest):,} bytes)')
            break

# Convert using gemmi command line (works better than Python API)
os.makedirs('structures/mmcif', exist_ok=True)
for pdb in glob.glob('structures/pdb_final/*.pdb'):
    name = os.path.basename(pdb).replace('.pdb','')
    cif = f'structures/mmcif/{name}.cif'
    ret = os.system(f'gemmi convert {pdb} {cif}')
    if ret == 0 and os.path.exists(cif):
        print(f'  Converted: {name}.cif')
    else:
        # MutPred-PPI also accepts PDB directly
        shutil.copy(pdb, f'structures/mmcif/{name}.pdb')
        print(f'  Fallback PDB: {name}.pdb')

print(f'\nFiles in mmcif/:')
!ls -la structures/mmcif/


Files in mmcif/:
total 8
drwxr-xr-x 2 root root 4096 Mar 16 14:56 .
drwxr-xr-x 5 root root 4096 Mar 16 14:56 ..


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3B: ALTERNATIVE — If Boltz-2 fails, use AF3 server
# ═══════════════════════════════════════════════════════════
# Skip this cell if Boltz-2 worked above.
#
# If Boltz-2 had memory/install issues:
# 1. Go to https://alphafoldserver.com
# 2. For each complex, copy-paste the JSON from structures/boltz_input/*_af3.json
# 3. Download the mmCIF results
# 4. Upload them here:

# from google.colab import files
# print('Upload AF3 mmCIF files:')
# uploaded_cifs = files.upload()
# os.makedirs('structures/af3_output', exist_ok=True)
# for fname in uploaded_cifs:
#     os.rename(fname, f'structures/af3_output/{fname}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4: CONVERT PDB TO mmCIF FOR MutPred-PPI
# ═══════════════════════════════════════════════════════════
import gemmi
import os, glob

# MutPred-PPI expects mmCIF format
os.makedirs('structures/mmcif', exist_ok=True)

pdb_files = glob.glob('structures/boltz_output/**/*.pdb', recursive=True)
print(f'PDB files to convert: {len(pdb_files)}')

for pdb_file in pdb_files:
    name = os.path.basename(os.path.dirname(pdb_file))
    if 'predictions' in pdb_file:
        name = os.path.basename(os.path.dirname(os.path.dirname(pdb_file)))

    cif_file = f'structures/mmcif/{name}.cif'

    try:
        structure = gemmi.read_structure(pdb_file)
        structure.write_mmcif(gemmi.MmcifDocument())
        # Simple conversion
        os.system(f'gemmi convert "{pdb_file}" "{cif_file}"')
        print(f'  {name}: converted to mmCIF')
    except Exception as e:
        print(f'  {name}: conversion error — {e}')
        # Fallback: just copy PDB, MutPred-PPI might handle it
        import shutil
        shutil.copy(pdb_file, f'structures/mmcif/{name}.pdb')
        print(f'  {name}: copied PDB as fallback')

print(f'\n✅ Structures ready in structures/mmcif/')
!ls -la structures/mmcif/

PDB files to convert: 6
  predictions: conversion error — 'gemmi.Structure' object has no attribute 'write_mmcif'
  predictions: copied PDB as fallback
  predictions: conversion error — 'gemmi.Structure' object has no attribute 'write_mmcif'
  predictions: copied PDB as fallback
  predictions: conversion error — 'gemmi.Structure' object has no attribute 'write_mmcif'
  predictions: copied PDB as fallback
  predictions: conversion error — 'gemmi.Structure' object has no attribute 'write_mmcif'
  predictions: copied PDB as fallback
  predictions: conversion error — 'gemmi.Structure' object has no attribute 'write_mmcif'
  predictions: copied PDB as fallback
  predictions: conversion error — 'gemmi.Structure' object has no attribute 'write_mmcif'
  predictions: copied PDB as fallback

✅ Structures ready in structures/mmcif/
total 308
drwxr-xr-x 2 root root   4096 Mar 16 09:34 .
drwxr-xr-x 7 root root   4096 Mar 16 09:36 ..
-rw-r--r-- 1 root root 305126 Mar 16 09:36 predictions.pdb


In [ ]:
import os, glob, shutil

# Fix: copy PDBs with correct names from Boltz output
os.makedirs('structures/pdb_final', exist_ok=True)

for pdb in glob.glob('structures/boltz_output/*/predictions/**/*.pdb', recursive=True):
    # Get complex name from path
    parts = pdb.split('/')
    for p in parts:
        if p in ['ATM_NBN','BRCA1_BARD1','BRCA1_PALB2','BRCA2_RAD51','MRE11_NBN','PALB2_BRCA2']:
            dest = f'structures/pdb_final/{p}.pdb'
            shutil.copy(pdb, dest)
            print(f'  {p}.pdb ({os.path.getsize(dest):,} bytes)')
            break

# Convert using gemmi command line (works better than Python API)
os.makedirs('structures/mmcif', exist_ok=True)
for pdb in glob.glob('structures/pdb_final/*.pdb'):
    name = os.path.basename(pdb).replace('.pdb','')
    cif = f'structures/mmcif/{name}.cif'
    ret = os.system(f'gemmi convert {pdb} {cif}')
    if ret == 0 and os.path.exists(cif):
        print(f'  Converted: {name}.cif')
    else:
        # MutPred-PPI also accepts PDB directly
        shutil.copy(pdb, f'structures/mmcif/{name}.pdb')
        print(f'  Fallback PDB: {name}.pdb')

print(f'\nFiles in mmcif/:')
!ls -la structures/mmcif/


Files in mmcif/:
total 308
drwxr-xr-x 2 root root   4096 Mar 16 09:34 .
drwxr-xr-x 7 root root   4096 Mar 16 09:36 ..
-rw-r--r-- 1 root root 305126 Mar 16 09:36 predictions.pdb


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5: RUN MutPred-PPI
# ═══════════════════════════════════════════════════════════
import os

os.chdir('/content/mutpred_ppi_workspace')

# Step 1: Generate contact graphs
print('Step 1: Generating contact graphs...')
!python /content/MutPred-PPI/src/01_make_contact_graphs_and_fasta.py \
    . \
    /content/structures/mmcif/ \
    triplets.tsv \
    4

print('\nStep 2: Running MutPred-PPI inference...')
!python /content/MutPred-PPI/src/02_run_mutpred-ppi_inference.py \
    . \
    --device cuda:0

print('\n✅ MutPred-PPI complete!')
!ls -la results/ 2>/dev/null || echo 'Results directory not found — check output above for errors'

os.chdir('/content')

Step 1: Generating contact graphs...
Traceback (most recent call last):
  File "/content/MutPred-PPI/src/01_make_contact_graphs_and_fasta.py", line 368, in <module>
    variant_indices, complex_ids, n_variant_partner_interactions = get_labeled_residues(variants_file)
                                                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/MutPred-PPI/src/01_make_contact_graphs_and_fasta.py", line 73, in get_labeled_residues
    res_idx = int(variant[1:-1]) - 1  # convert to 0-based
              ^^^^^^^^^^^^^^^^^^
ValueError: invalid literal for int() with base 10: '.979'

Step 2: Running MutPred-PPI inference...
Traceback (most recent call last):
  File "/content/MutPred-PPI/src/02_run_mutpred-ppi_inference.py", line 18, in <module>
    from utils.inference_utils import run_inference_on_dataset
  File "/content/MutPred-PPI/src/utils/inference_utils.py", line 15, in <module>
    from .model_loader import get_models, model_predict
  File "/

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6: ANALYZE RESULTS AND GENERATE FIGURES
# ═══════════════════════════════════════════════════════════

# Fix for numpy.dtype size changed error and missing torch_geometric
# Ideally, these installations should be in Cell 1, followed by a runtime restart.
# However, to address the error within this cell, we place them here.
!pip install --upgrade numpy pandas torch_geometric

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Load MutPred-PPI results
results_file = 'mutpred_ppi_workspace/results/MutPred-PPI_preds.tsv'
if os.path.exists(results_file):
    df_ppi = pd.read_csv(results_file, sep='\t')
    print(f'MutPred-PPI results: {len(df_ppi)} variant-partner pairs')
    print(f'Columns: {list(df_ppi.columns)}')
    print(f'\nScore distribution:')
    print(df_ppi['score'].describe())

    # Disrupting = score > 0.5
    df_ppi['disrupting'] = df_ppi['score'] > 0.5
    n_dis = df_ppi['disrupting'].sum()
    print(f'\nPPI-disrupting: {n_dis}/{len(df_ppi)} ({100*n_dis/len(df_ppi):.1f}%)')

    # Merge with variant metadata
    df_meta = pd.read_csv('mutpred_ppi_workspace/variant_triplets_full.csv')
    print(f'\nVariant metadata: {len(df_meta)} triplets')
    print(f'Biallelic: {df_meta["is_biallelic"].sum()}')

    # Compare disruption rates: biallelic vs non-biallelic
    # (This requires matching the PPI results back to variant metadata)

    # Figure: Score distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Panel A: Overall score distribution
    axes[0].hist(df_ppi['score'], bins=50, color='#3498DB', alpha=0.7, edgecolor='white')
    axes[0].axvline(0.5, color='#E74C3C', linestyle='--', linewidth=2, label='Disruption threshold')
    axes[0].set_xlabel('MutPred-PPI Score', fontsize=11)
    axes[0].set_ylabel('Count', fontsize=11)
    axes[0].set_title('A. PPI Disruption Score Distribution', fontsize=12, fontweight='bold')
    axes[0].legend()

    # Panel B: Per-complex disruption rate
    if 'complex' in df_ppi.columns:
        comp_rates = df_ppi.groupby('complex')['disrupting'].mean()
        comp_rates.plot.barh(ax=axes[1], color='#E74C3C', alpha=0.7)
        axes[1].set_xlabel('Fraction PPI-disrupting', fontsize=11)
        axes[1].set_title('B. Disruption Rate by Complex', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.savefig('Fig11_mutpred_ppi_results.png', dpi=300)
    plt.show()
    print('\nSaved: Fig11_mutpred_ppi_results.png')

else:
    print('❌ MutPred-PPI results not found.')
    print('   Check Cell 5 output for errors.')
    print('   Common issues:')
    print('   - Structure files not in expected format')
    print('   - Missing dependencies')
    print('   - Variant format mismatch')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.1 MB/s eta 0:00:00
  Using cached torch_geometric-2.7.0-py3-none-any.whl.metadata (63 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 125.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.0 requires numpy<2.2,>=1.24, but you have numpy 2.4.3 which is incompatible.
boltz 2.2.1 requires numpy<2.0,>=1.26, but you

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7: DOWNLOAD RESULTS
# ═══════════════════════════════════════════════════════════
import shutil

# Package all results
!tar czf alphahrd_structural_results.tar.gz \
    structures/boltz_output/ \
    structures/mmcif/ \
    mutpred_ppi_workspace/results/ \
    Fig11_mutpred_ppi_results.png \
    2>/dev/null

from google.colab import files
files.download('alphahrd_structural_results.tar.gz')
print('\n✅ Results packaged and downloaded!')
print('Upload alphahrd_structural_results.tar.gz to the next Claude conversation')
print('to generate the final manuscript figures.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Results packaged and downloaded!
Upload alphahrd_structural_results.tar.gz to the next Claude conversation
to generate the final manuscript figures.


In [ ]:
import os, glob

# Find all PDB files
pdbs = glob.glob('structures/boltz_output/**/*.pdb', recursive=True)
print(f'PDB files found: {len(pdbs)}')
for p in pdbs:
    print(f'  {p} ({os.path.getsize(p):,} bytes)')

# Package them
!tar czf boltz2_structures.tar.gz structures/boltz_output/
from google.colab import files
files.download('boltz2_structures.tar.gz')


PDB files found: 6
  structures/boltz_output/MRE11_NBN/boltz_results_MRE11_NBN/predictions/MRE11_NBN/MRE11_NBN_model_0.pdb (305,855 bytes)
  structures/boltz_output/PALB2_BRCA2/boltz_results_PALB2_BRCA2/predictions/PALB2_BRCA2/PALB2_BRCA2_model_0.pdb (221,534 bytes)
  structures/boltz_output/BRCA1_BARD1/boltz_results_BRCA1_BARD1/predictions/BRCA1_BARD1/BRCA1_BARD1_model_0.pdb (143,288 bytes)
  structures/boltz_output/BRCA1_PALB2/boltz_results_BRCA1_PALB2/predictions/BRCA1_PALB2/BRCA1_PALB2_model_0.pdb (65,690 bytes)
  structures/boltz_output/BRCA2_RAD51/boltz_results_BRCA2_RAD51/predictions/BRCA2_RAD51/BRCA2_RAD51_model_0.pdb (172,772 bytes)
  structures/boltz_output/ATM_NBN/boltz_results_ATM_NBN/predictions/ATM_NBN/ATM_NBN_model_0.pdb (305,126 bytes)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os, glob
import numpy as np
import pandas as pd
from collections import defaultdict

# Parse PDB: extract CA atoms per chain
def parse_pdb(pdb_file):
    chains = defaultdict(list)
    with open(pdb_file) as f:
        for line in f:
            if line.startswith('ATOM') and line[12:16].strip() == 'CA':
                chain = line[21]
                resid = int(line[22:26].strip())
                resname = line[17:20].strip()
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                chains[chain].append({'resid': resid, 'resname': resname, 'x': x, 'y': y, 'z': z})
    return chains

# Find interface residues (CA-CA distance < 8A between chains)
INTERFACE_CUTOFF = 8.0

pdbs = sorted(glob.glob('structures/boltz_output/**/*.pdb', recursive=True))
results = []

for pdb_file in pdbs:
    name = os.path.basename(pdb_file).replace('_model_0.pdb', '')
    chains = parse_pdb(pdb_file)
    chain_ids = sorted(chains.keys())

    if len(chain_ids) < 2:
        print(f'{name}: only {len(chain_ids)} chain(s), skipping')
        continue

    chain_a, chain_b = chain_ids[0], chain_ids[1]
    res_a = chains[chain_a]
    res_b = chains[chain_b]

    # Compute pairwise distances
    interface_a = set()
    interface_b = set()
    min_distances = {}

    for ra in res_a:
        for rb in res_b:
            d = np.sqrt((ra['x']-rb['x'])**2 + (ra['y']-rb['y'])**2 + (ra['z']-rb['z'])**2)
            key_a = (chain_a, ra['resid'])
            key_b = (chain_b, rb['resid'])

            if key_a not in min_distances or d < min_distances[key_a]:
                min_distances[key_a] = d
            if key_b not in min_distances or d < min_distances[key_b]:
                min_distances[key_b] = d

            if d < INTERFACE_CUTOFF:
                interface_a.add(ra['resid'])
                interface_b.add(rb['resid'])

    n_a = len(res_a)
    n_b = len(res_b)
    n_int_a = len(interface_a)
    n_int_b = len(interface_b)

    print(f'\n{name}:')
    print(f'  Chain {chain_a}: {n_a} residues, {n_int_a} at interface ({100*n_int_a/max(n_a,1):.1f}%)')
    print(f'  Chain {chain_b}: {n_b} residues, {n_int_b} at interface ({100*n_int_b/max(n_b,1):.1f}%)')

    results.append({
        'complex': name,
        'chain_a': chain_a, 'n_res_a': n_a, 'n_interface_a': n_int_a,
        'chain_b': chain_b, 'n_res_b': n_b, 'n_interface_b': n_int_b,
        'interface_residues_a': sorted(interface_a),
        'interface_residues_b': sorted(interface_b),
    })

df_interface = pd.DataFrame(results)
df_interface.to_csv('boltz2_interface_analysis.csv', index=False)

# Also save per-residue distances for variant mapping
all_distances = []
for pdb_file in pdbs:
    name = os.path.basename(pdb_file).replace('_model_0.pdb', '')
    chains = parse_pdb(pdb_file)
    chain_ids = sorted(chains.keys())
    if len(chain_ids) < 2: continue

    for res in chains[chain_ids[0]]:
        min_d = min(np.sqrt((res['x']-rb['x'])**2+(res['y']-rb['y'])**2+(res['z']-rb['z'])**2)
                    for rb in chains[chain_ids[1]])
        all_distances.append({
            'complex': name, 'chain': chain_ids[0],
            'resid': res['resid'], 'resname': res['resname'],
            'min_dist_to_partner': round(min_d, 2),
            'at_interface': min_d < INTERFACE_CUTOFF
        })
    for res in chains[chain_ids[1]]:
        min_d = min(np.sqrt((res['x']-ra['x'])**2+(res['y']-ra['y'])**2+(res['z']-ra['z'])**2)
                    for ra in chains[chain_ids[0]])
        all_distances.append({
            'complex': name, 'chain': chain_ids[1],
            'resid': res['resid'], 'resname': res['resname'],
            'min_dist_to_partner': round(min_d, 2),
            'at_interface': min_d < INTERFACE_CUTOFF
        })

df_dist = pd.DataFrame(all_distances)
df_dist.to_csv('boltz2_residue_distances.csv', index=False)

print(f'\n\nSaved: boltz2_interface_analysis.csv')
print(f'Saved: boltz2_residue_distances.csv ({len(df_dist)} residues)')

# Download
from google.colab import files
files.download('boltz2_interface_analysis.csv')
files.download('boltz2_residue_distances.csv')


ATM_NBN:
  Chain A: 445 residues, 18 at interface (4.0%)
  Chain B: 21 residues, 13 at interface (61.9%)

BRCA1_BARD1:
  Chain A: 109 residues, 24 at interface (22.0%)
  Chain B: 115 residues, 23 at interface (20.0%)

BRCA1_PALB2:
  Chain A: 65 residues, 21 at interface (32.3%)
  Chain B: 36 residues, 18 at interface (50.0%)

BRCA2_RAD51:
  Chain A: 35 residues, 18 at interface (51.4%)
  Chain B: 243 residues, 22 at interface (9.1%)

MRE11_NBN:
  Chain A: 411 residues, 38 at interface (9.2%)
  Chain B: 56 residues, 28 at interface (50.0%)

PALB2_BRCA2:
  Chain A: 334 residues, 19 at interface (5.7%)
  Chain B: 19 residues, 14 at interface (73.7%)


Saved: boltz2_interface_analysis.csv
Saved: boltz2_residue_distances.csv (1889 residues)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>